# 02-1. LangGraph State, Node, Edge

- 예상 시간: 120분
- 선수 실습: 01-4_agent_loop_react.ipynb
- 실습 난이도: 기초~중급
- 핵심 기술: StateGraph, TypedDict, Node, Edge
- 최종 산출물: 기본 LangGraph Workflow
- 버전: 학생용 실습파일 (`TODO`가 표시된 코드 셀을 완성해야 이후 셀이 정상 동작합니다)

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. LangGraph State의 역할을 설명할 수 있다.
2. Node 함수를 작성하고 State를 갱신할 수 있다.
3. Edge를 이용해 Node 실행 순서를 연결할 수 있다.
4. Node 실행 전후 State 변화를 확인할 수 있다.

## 2. 문제 상황

Notebook 01-4에서는 `state = {"steps": [], "observations": [], ...}` 형태의 딕셔너리를
직접 만들고, 반복문 안에서 직접 갱신했다. 이 방식은 단계가 2~3개일 때는 괜찮지만,
Node 수가 늘어나고 조건에 따라 실행 순서가 달라지기 시작하면 "어떤 함수가 State의
어떤 필드를 읽고 쓰는지"를 추적하기 어려워진다. LangGraph는 State(공유 데이터),
Node(작업 단계), Edge(실행 순서)를 명시적인 구조로 분리해서 이 문제를 해결한다.

> 실행 단계가 늘어날수록, 상태를 직접 관리하는 코드보다 State/Node/Edge로 구조화된
> 코드가 흐름을 파악하고 수정하기 쉽다.

## 3. 핵심 개념

### 3.1 State란 무엇인가

State는 LangGraph Workflow의 여러 Node가 공유하는 작업 데이터이다. 사용자 입력,
중간 분석 결과, Tool 실행 결과, 오류 상태와 완료 여부 등을 State에 저장하여 다음
Node에 전달한다. 각 Node는 State 전체를 새로 만드는 것이 아니라, 자신이 변경할
필드만 반환하면 LangGraph가 나머지 필드와 병합해 준다.

### 3.2 개념이 필요한 이유

일반적인 함수 호출만으로 순차 실행은 가능하지만, 실행 단계가 많아지면 중간 결과와
오류 상태를 일관되게 전달하기 어렵다. Notebook 01-4에서 직접 만든 `state` 딕셔너리를
여러 함수에 인자로 넘기고 다시 받는 방식이 그 예이다. State를 Schema로 명시하면
Workflow 전체에서 공유해야 하는 데이터를 한 곳에서 관리할 수 있다.

### 3.3 주요 구성요소

| 구성요소 | 역할 |
|---|---|
| State | 여러 Node가 공유하는 데이터. `TypedDict`로 필드와 타입을 명시한다 |
| Node | State를 입력받아 작업을 수행하고, 변경할 필드만 반환하는 함수 |
| Edge | Node의 실행 순서를 정의하는 경로 (데이터를 저장하지 않는다) |
| START / END | Graph의 시작 지점과 종료 지점을 나타내는 특수 표식 |
| Compile / Invoke | 설계(Compile)와 실행(Invoke)을 분리하는 두 단계 |

Compile은 `add_node`/`add_edge`로 그린 설계도를 실행 가능한 객체로 굳히는 단계이고,
Invoke는 그 객체에 초기 State를 넣어 실제로 한 번 실행하는 단계이다.

### 3.4 동작 과정

```text
START
  ↓
analyze_input   (State 중 input_type, analysis 필드만 갱신)
  ↓
generate_answer (State 중 final_answer 필드만 갱신)
  ↓
END
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `class AgentState(TypedDict)` | Graph에서 공유할 State Schema |
| `builder.add_node("analyze_input", analyze_input)` | 실행 Node 등록 |
| `builder.add_edge(START, "analyze_input")` | 고정 실행 경로 연결 |
| `builder.compile()` | 설계도를 실행 가능한 Graph로 변환 |
| `graph.invoke(initial_state)` | Graph 실행 |

### 3.6 유사 개념과의 차이

**State vs 전역변수**

| 구분 | 전역변수 | LangGraph State |
|---|---|---|
| 접근 범위 | 어디서나 임의로 읽고 쓸 수 있음 | 정의된 Schema의 필드만 Node를 통해 갱신 |
| 변경 이력 | 누가 언제 바꿨는지 추적 어려움 | 어떤 Node가 어떤 필드를 반환했는지 명확 |
| 실행 간 독립성 | 여러 실행이 상태를 공유해 섞일 위험 | `invoke()` 호출마다 독립된 State 사용 |

**LLM Message Context vs Graph State**

| 구분 | LLM Message Context | Graph State |
|---|---|---|
| 저장 대상 | System/Human/AI/Tool Message 기록 | Workflow 전체가 공유하는 임의의 구조화 데이터 |
| Schema | 메시지 리스트 형태로 비교적 고정 | `TypedDict`로 자유롭게 정의 |
| 갱신 방식 | 메시지를 계속 追加 | Node가 반환한 필드로 병합 |
| 범위 | 하나의 LLM 호출 흐름 | 여러 Node·여러 LLM 호출을 아우르는 전체 실행 |

### 3.7 사용 시점과 적용 조건

Workflow 단계가 여러 개이고, 각 단계가 이전 단계의 결과에 의존하며, 앞으로 분기나
재사용 가능성이 있는 경우에 LangGraph를 사용한다. 단계가 1~2개뿐이고 분기가 없다면
일반 함수 호출로도 충분하다.

### 3.8 한계와 주의사항

- Node가 반환하는 키 이름이 State Schema의 필드명과 정확히 일치해야 한다. 오타가
  있으면 오류 없이 조용히 그 필드가 갱신되지 않을 수 있다.
- Node/Edge 수가 늘어날수록 전체 흐름을 한눈에 파악하기 어려워지므로, Graph 구조를
  문서화하거나 시각화하는 습관이 필요하다.

### 3.9 자주 발생하는 오해

- "Node는 State 전체를 새로 만들어 반환해야 한다"는 오해가 있다. 실제로는 변경할
  필드만 반환하면 되고, 나머지 필드는 LangGraph가 기존 값을 유지한다.
- "Edge는 데이터를 저장하는 객체이다"라는 오해도 흔하다. Edge는 실행 경로만
  정의하며, 실제 데이터는 State가 담당한다.

### 3.10 핵심 정리

- State는 여러 Node가 공유하는 데이터이며, Node는 변경할 필드만 반환한다.
- Edge는 실행 경로를 정의할 뿐 데이터를 저장하지 않는다.
- Compile은 설계를 실행 가능하게 만들고, Invoke는 그것을 실제로 실행한다.
- 키 이름이 State Schema와 어긋나면 오류 없이 조용히 실패할 수 있으므로 주의해야
  한다.

## 4. 실행 구조

```text
build_graph()
   │
   ├─ StateGraph(AgentState) 생성
   ├─ add_node("analyze_input", analyze_input)
   ├─ add_node("generate_answer", generate_answer)
   ├─ add_edge(START, "analyze_input")
   ├─ add_edge("analyze_input", "generate_answer")
   ├─ add_edge("generate_answer", END)
   └─ compile() → 실행 가능한 Graph

graph.invoke(initial_state) → 최종 State
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [ ]:
from langgraph.graph import END, START, StateGraph

from agentic_ai.logging_utils import save_log
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import OUTPUT_DIR, PROJECT_ROOT
from agentic_ai.states import AgentState

print_environment_summary()


## 6. 최소 실행 예제

Node가 하나뿐인 가장 단순한 Graph로 Compile/Invoke의 흐름을 먼저 확인한다.

In [ ]:
def echo_node(state: AgentState) -> dict:
    """입력을 그대로 반영하는 최소 Node. final_answer 필드만 갱신한다."""
    return {"final_answer": f"받은 입력: {state['user_input']}"}


mini_builder = StateGraph(AgentState)
mini_builder.add_node("echo", echo_node)
mini_builder.add_edge(START, "echo")
mini_builder.add_edge("echo", END)
mini_graph = mini_builder.compile()

mini_initial_state = {
    "user_input": "안녕하세요",
    "input_type": "",
    "analysis": {},
    "final_answer": "",
    "error": None,
}
mini_result = mini_graph.invoke(mini_initial_state)
print(mini_result)

## 7. 단계별 구현

### 7.1 State 정의

`AgentState`는 작업지시서가 요구하는 스키마 그대로 `src/agentic_ai/states.py`에
정의되어 있다. 여러 Notebook에서 재사용할 수 있도록 공통 모듈로 분리했다.

```python
class AgentState(TypedDict):
    user_input: str
    input_type: str
    analysis: dict
    final_answer: str
    error: str | None
```

### 7.2 Node 함수 작성

`analyze_input`은 규칙 기반으로 입력 유형을 판단하고, `generate_answer`는 그 결과를
바탕으로 LLM 응답을 생성한다. 두 Node 모두 State 전체가 아니라 자신이 바꿀 필드만
반환한다.

In [ ]:
def analyze_input(state: AgentState) -> dict:
    """사용자 입력을 규칙 기반으로 간단히 분석한다."""
    text = state["user_input"]
    has_number = any(ch.isdigit() for ch in text)
    input_type = "calculation" if has_number else "question"
    analysis = {"length": len(text), "has_number": has_number}
    return {"input_type": input_type, "analysis": analysis}


def generate_answer(state: AgentState) -> dict:
    """이 실습은 Graph 구조에 집중하므로 LLM 없이 분류 결과를 확인한다."""
    return {
        "final_answer": f"[{state['input_type']}] {state['user_input']}",
        "error": None,
    }


print(analyze_input({"user_input": "128 나누기 4는?", "input_type": "", "analysis": {}, "final_answer": "", "error": None}))


### 7.3 Graph 생성, Node 등록, Edge 연결, Compile

**TODO**: `build_graph()`를 작성한다.

- `StateGraph(AgentState)`로 builder를 만든다.
- `add_node()`로 `"analyze_input"`과 `"generate_answer"`를 등록한다.
- `add_edge()`로 `START → analyze_input → generate_answer → END`를 연결한다.
- `compile()`한 결과를 반환한다.

예상 출력: `build_graph()`가 `invoke()` 가능한 Graph 객체를 반환한다.

In [ ]:
def build_graph():
    """analyze_input -> generate_answer 순서로 실행되는 기본 Graph를 만든다."""
    # TODO: 1) StateGraph(AgentState)로 builder를 만든다.
    # TODO: 2) add_node()로 "analyze_input"과 "generate_answer"를 등록한다.
    # TODO: 3) add_edge()로 START -> analyze_input -> generate_answer -> END를
    #       연결한다.
    # TODO: 4) builder.compile()한 결과를 반환한다.
    raise NotImplementedError("TODO: build_graph를 완성하세요.")


graph = build_graph()
print(graph)


In [ ]:
# 그래프 그리기

from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

### 7.4 Invoke

초기 State를 만들어 Graph를 실행한다.

In [ ]:
initial_state = {
    "user_input": "128 나누기 4는 얼마야?",
    "input_type": "",
    "analysis": {},
    "final_answer": "",
    "error": None,
}
result = graph.invoke(initial_state)
print(result)

## 8. 실행 결과 관찰

실행 전(`initial_state`)과 실행 후(`result`)의 State를 나란히 비교해서 어떤 필드가
어떤 Node에 의해 바뀌었는지 확인한다.

In [ ]:
for key in initial_state:
    before = initial_state[key]
    after = result[key]
    changed = "변경됨" if before != after else "변경 없음"
    print(f"{key:12s} | 이전: {before!r:30} | 이후: {after!r:30} | {changed}")

**결과 해석**: `input_type`과 `analysis`는 `analyze_input`이 갱신하고, `final_answer`는
`generate_answer`가 갱신한다. `user_input`은 어느 Node도 반환하지 않았으므로 원래
값 그대로 유지된다.

## 9. 실패 실험: 키 이름 불일치

Node가 State Schema와 다른 키 이름을 반환하면 어떤 일이 벌어지는지 확인한다.

In [ ]:
def broken_analyze_input(state: AgentState) -> dict:
    """input_type 대신 inputType이라는 잘못된 키를 반환하는 예시."""
    text = state["user_input"]
    has_number = any(ch.isdigit() for ch in text)
    return {"inputType": "calculation" if has_number else "question"}


broken_builder = StateGraph(AgentState)
broken_builder.add_node("analyze_input", broken_analyze_input)
broken_builder.add_node("generate_answer", generate_answer)
broken_builder.add_edge(START, "analyze_input")
broken_builder.add_edge("analyze_input", "generate_answer")
broken_builder.add_edge("generate_answer", END)
broken_graph = broken_builder.compile()



In [ ]:
# 시각화는 네트워크 환경에 따라 실패할 수 있으므로 선택적으로 실행한다.
from IPython.display import Image, display

try:
    display(Image(broken_graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("그래프 시각화를 건너뜁니다:", exc)


In [ ]:
# 실행
broken_initial = {
    "user_input": "128 나누기 4는 얼마야?",
    "input_type": "unknown",
    "analysis": {},
    "final_answer": "",
    "error": None,
}
broken_result = broken_graph.invoke(broken_initial)
print(broken_result)
print("input_type이 바뀌었는가:", broken_result["input_type"] != broken_initial["input_type"])

**원인 분석 질문**

- 코드는 오류 없이 끝까지 실행되었는가?
- `input_type` 필드는 왜 `"unknown"` 그대로인가?
- `generate_answer`는 잘못된 `input_type`을 받은 채로 계속 실행되었는가?
- 이런 문제를 코드 실행 중 오류 메시지만으로 발견할 수 있는가?

## 10. 오류 수정 실습

Node가 반환한 키가 State Schema에 있는 필드인지 미리 검사하는 도구를 만들어, 이런
실수를 실행 전에 발견할 수 있게 한다.

**TODO**: `validate_node_output()`을 작성한다.

- `update` 딕셔너리의 키 중 `allowed_keys`에 없는 키의 목록을 리스트로 반환한다.
- 잘못된 키가 없으면 빈 리스트를 반환한다.

In [ ]:
def validate_node_output(update: dict, allowed_keys: set[str]) -> list[str]:
    """Node가 반환한 dict의 키가 허용된 State 필드에 속하는지 검사한다."""
    # TODO: update의 키 중 allowed_keys에 없는 키만 리스트로 반환한다.
    #       잘못된 키가 없으면 빈 리스트를 반환한다.
    raise NotImplementedError("TODO: validate_node_output을 완성하세요.")


allowed_keys = set(AgentState.__annotations__.keys())

bad_update = broken_analyze_input(broken_initial)
print("잘못된 키:", validate_node_output(bad_update, allowed_keys))

good_update = analyze_input(broken_initial)
print("잘못된 키:", validate_node_output(good_update, allowed_keys))


**수정 결과 재검증**: 잘못된 Node의 출력에서는 `["inputType"]`이 감지되고, 올바른
Node의 출력에서는 빈 리스트가 나와야 한다.

In [ ]:
assert validate_node_output({"inputType": "x"}, allowed_keys) == ["inputType"]
assert validate_node_output({"input_type": "x"}, allowed_keys) == []
print("재검증 통과")

## 11. 도전 과제

1. `analyze_input`과 `generate_answer` 사이에 로그를 남기는 `log_node`를 추가해서
   3단계 Graph로 확장해본다.
2. `initial_state`에서 `user_input`을 빈 문자열로 바꾸고 실행했을 때 각 Node가 어떻게
   반응하는지 관찰한다.
3. `validate_node_output()`을 Node 실행 직후 자동으로 호출하도록 `build_graph()`를
   감싸는 디버그 유틸리티를 만들어본다.

## 12. 테스트

**테스트 유형: 단위·로컬 Graph 구조 테스트 — 결정적, 외부 API 호출 없음**

LLM 호출 없이 검증 가능한 부분을 테스트한다.

In [ ]:
calc_state = {"user_input": "128 나누기 4", "input_type": "", "analysis": {}, "final_answer": "", "error": None}
calc_update = analyze_input(calc_state)
assert calc_update["input_type"] == "calculation"
assert calc_update["analysis"]["has_number"] is True
assert calc_update["analysis"]["length"] == len(calc_state["user_input"])

question_state = {"user_input": "오늘 날씨 어때", "input_type": "", "analysis": {}, "final_answer": "", "error": None}
assert analyze_input(question_state)["input_type"] == "question"

assert validate_node_output({"inputType": "x"}, allowed_keys) == ["inputType"]
assert validate_node_output({"input_type": "x"}, allowed_keys) == []

graph_instance = build_graph()
assert graph_instance is not None
print("테스트 통과")

## 13. 결과 저장

In [ ]:
state_node_edge_log = {
    "initial_state": initial_state,
    "result_state": result,
    "broken_result_state": broken_result,
    "invalid_keys_detected": validate_node_output(bad_update, allowed_keys),
}
saved_path = save_log(state_node_edge_log, OUTPUT_DIR / "logs" / "05_state_node_edge_log.json")
print("저장 위치:", saved_path)

## 14. 핵심 정리

- State는 여러 Node가 공유하는 데이터이며, 전역변수와 달리 정해진 Schema의 필드만
  Node를 통해 갱신된다.
- Node는 State 전체가 아니라 자신이 변경할 필드만 반환하면 된다.
- Edge는 실행 경로만 정의할 뿐 데이터를 저장하지 않는다.
- Compile은 설계도를 실행 가능하게 만들고, Invoke는 그것을 실제로 실행한다.
- Node가 반환하는 키 이름이 State Schema와 어긋나면 오류 없이 조용히 실패할 수
  있으므로, 키 이름을 검증하는 습관이 필요하다.

## 15. 확인 문제

1. Node는 State 전체를 반환해야 하는가, 아니면 변경할 필드만 반환해도 되는가?
2. State의 일부 필드만 수정하고 싶을 때 Node는 어떻게 작성해야 하는가?
3. Node가 반환하는 키 이름이 State Schema와 다르면 어떤 문제가 발생하는가?
4. `START`와 `END`는 각각 어떤 역할을 하는가?